# **4일차 과제 — 진료 데이터 기반 벡터DB 구축하기**

* **배경 시나리오**: OO 병원에서 고혈압·당뇨 등 만성질환자를 위한 AI 챗봇 서비스를 기획 중입니다. 오늘 과제로 이 챗봇의 지식 기반이 될 벡터DB를 구축합니다. 여기서 만든 벡터DB는 5·6일차 과제에서 그대로 이어서 사용합니다.

> **제출자: 조태희** · OZ코딩스쿨 AI 헬스케어 부트캠프 6기 · 자연어 처리(NLP) 4일차 과제 (RAG 기초1)

**📌 실행 안내**
- 실행 번호와 출력이 있는 셀은 API 키 없이도 돌아가는 셀(질환·관점 정의, State·노드 함수 정의, 검색어 미리보기, 그래프 조립·구조 확인)이라 미리 실행해 둔 결과입니다.
- 코드 첫 줄에 **🔑 [Colab에서 실행]** 표시가 있는 셀은 **OpenAI API 키와 Tavily API 키**가 필요해서 아직 실행하지 않았습니다. Colab에서 `api_key.txt`(두 키가 한 줄씩)를 업로드하거나 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY`, `TAVILY_API_KEY`를 등록한 뒤 **런타임 → 모두 실행**으로 위에서부터 차례대로 실행해 주세요.
- 데이터 수집(검색 18회 + 정제 최대 54회)에 몇 분 정도 걸립니다. 검색 결과는 실행 시점마다 달라질 수 있습니다.
- 여기서 만든 `chronic_db`는 5·6일차에서 그대로 씁니다. 마지막 **6. (추가)** 셀로 구글 드라이브 `ai_agent` 폴더에 복사해 두세요.

# **1. 환경준비**

## **(1) 라이브러리**

In [ ]:
# 🔑 [Colab에서 실행] 필요한 라이브러리 설치 (-q: 설치 로그를 짧게 표시)
!pip install langchain-openai langchain-tavily langgraph langchain-community langchain-chroma langchain-text-splitters -q

In [1]:
import os                                 # 환경변수(API 키), 파일 경로 관리
import warnings                           # 경고 메시지 설정 도구
# langchain-community 패키지가 "유지보수 종료 예정"이라는 안내(DeprecationWarning)를 띄우는데,
# 지금 쓰는 TextLoader 동작에는 문제가 없어서 그 안내 문구만 숨깁니다.
warnings.filterwarnings("ignore", message=".*langchain-community.*", category=DeprecationWarning)
from typing import TypedDict              # State의 칸(키)과 타입을 정의하는 도구
from collections import defaultdict       # 없는 키를 처음 쓸 때 기본값(빈 딕셔너리 등)을 자동으로 만들어 주는 딕셔너리

from langchain_openai import ChatOpenAI, OpenAIEmbeddings        # GPT 모델 / 문장을 벡터(숫자 목록)로 바꾸는 임베딩 모델
from langchain_core.messages import HumanMessage, SystemMessage  # 메시지 종류
from langchain_tavily import TavilySearch                        # Tavily 웹 검색 도구
from langchain_community.document_loaders import TextLoader      # 텍스트 파일 → Document 객체로 읽어오는 로더
from langchain_text_splitters import CharacterTextSplitter       # 긴 문서를 청크(조각)로 자르는 도구
from langchain_chroma import Chroma                              # 벡터DB(Chroma)
from langgraph.graph import StateGraph, START, END               # 그래프 설계도, 시작점, 끝점

## **(2) API Key 확인**

In [ ]:
# 🔑 [Colab에서 실행] api_key.txt 파일 업로드
#   - api_key.txt 안에는 아래처럼 한 줄에 하나씩 적혀 있어야 합니다.
#       OPENAI_API_KEY=sk-...
#       TAVILY_API_KEY=tvly-...
#   - Colab 왼쪽 🔑(보안 비밀)에 두 키를 등록해 두었다면 이 셀은 건너뛰어도 됩니다.
from google.colab import files   # Colab에서 내 컴퓨터 파일을 올리는 기능
uploaded = files.upload()        # 업로드 창이 뜨면 api_key.txt를 선택합니다

In [ ]:
# 🔑 [Colab에서 실행] API 키를 환경변수로 등록합니다 (코드에 키를 직접 적지 않기 위해서예요)
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:                      # 키 파일을 읽기 모드로 엽니다
        for line in f:                                  # 한 줄씩 읽으면서
            line = line.strip()                         # 앞뒤 공백·줄바꿈을 지우고
            if line and "=" in line:                    # 빈 줄이 아니고 "이름=값" 형태라면
                key, value = line.split("=", 1)         # 첫 번째 '='를 기준으로 이름과 값을 나눠서
                os.environ[key.strip()] = value.strip() # 환경변수로 등록합니다

def ensure_api_key(name):
    """환경변수에 키가 없으면 Colab 보안 비밀 → 직접 입력 순서로 채웁니다."""
    if os.environ.get(name):                            # 이미 등록돼 있으면 그대로 사용
        return                                          # 더 할 일이 없으니 함수 종료
    try:
        from google.colab import userdata               # Colab 🔑 보안 비밀 저장소
        os.environ[name] = userdata.get(name)           # 보안 비밀에 등록된 키를 꺼내 등록
    except Exception:                                   # 보안 비밀에 없거나 Colab이 아니면
        from getpass import getpass                     # 입력한 글자가 화면에 안 보이는 입력창
        os.environ[name] = getpass(f"{name}를 입력하세요: ")   # 직접 입력받아 등록


if os.path.exists("api_key.txt"):                       # 업로드한 키 파일이 있으면
    load_api_keys("api_key.txt")                        # 파일에서 읽어서 등록
ensure_api_key("OPENAI_API_KEY")                        # OpenAI 키 (정제 LLM + 임베딩)
ensure_api_key("TAVILY_API_KEY")                        # Tavily 키 (웹 검색)

* 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
# 🔑 [Colab에서 실행] 키가 잘 들어왔는지 앞부분만 확인합니다
#   (전체 키가 노트북에 남으면 유출될 수 있어서 앞 7글자만 보이게 했습니다. 확인 후 이 셀은 삭제하세요.)
print(os.environ['OPENAI_API_KEY'][:7] + "...")
print(os.environ['TAVILY_API_KEY'][:7] + "...")

In [ ]:
# 🔑 [Colab에서 실행] 모델·도구 준비 (객체를 만들 때 API 키를 확인합니다)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)               # 원문 정제에 쓸 GPT (temperature=0: 답 흔들림 최소화)
tavily_tool = TavilySearch(max_results=3, include_raw_content=True) # 검색어당 결과 3개, 웹페이지 원문(raw_content)까지 받기

# **2. 데이터 수집 Agent (검색 노드 → 정리 노드)**

* 질환 6개 × 관점 3개 = 총 18회 검색을 수행하고, 검색된 원문을 LLM으로 정제하는 2노드 파이프라인입니다.
* ⚠️ 검색 18회 + LLM 정제 최대 54회가 순차적으로 진행되어, 실행에 수 분 정도 소요될 수 있습니다.

**💡 풀이 포인트** — '자료 조사원'(`search_node`)이 웹에서 원문을 모아 State의 `raw_items` 칸에 넣으면, '편집자'(`clean_node`)가 그걸 꺼내 광고·메뉴 같은 잡음을 걷어내고 `clean_items` 칸에 넣는 2단계 구조입니다. Tavily 결과의 `raw_content`(웹페이지 원문 전체)는 페이지에 따라 비어 있을(None) 수 있어서, 비어 있으면 `content`(검색어와 관련된 부분만 짧게 뽑아 준 조각)를 대신 씁니다. 웹 원문은 아주 길 수 있어서 정제할 때는 앞부분 8,000자만 LLM에 보내 비용과 시간을 줄였습니다.

## **(1) 대상 질환·관점 정의**

In [2]:
DISEASE_LIST = ["고혈압", "제2형 당뇨병", "고지혈증", "골다공증", "천식", "만성신장질환"]   # 대상 질환 6개

ASPECTS = {                                                    # 관점 이름 → 검색어 틀 ({disease} 자리에 질환명이 들어감)
    "증상과 진단 기준": "{disease} 증상과 진단 기준",
    "생활습관 관리 방법": "{disease} 생활습관 관리 방법",
    "복약 및 주의사항": "{disease} 복약 및 주의사항",
}

## **(2) State 정의**

In [3]:
class PipelineState(TypedDict):
    raw_items: list      # [{'disease':.., 'aspect':.., 'raw_content':..}, ...]    ← 검색 노드가 채우는 칸
    clean_items: list    # [{'disease':.., 'aspect':.., 'clean_content':..}, ...]  ← 정리 노드가 채우는 칸

## **(3) 검색 노드**

In [4]:
def search_node(state: PipelineState):
    raw_items = []   # 검색 원문을 모아 둘 리스트

    for disease in DISEASE_LIST:                               # 질환 6개를 하나씩
        for aspect_name, query_template in ASPECTS.items():    # 관점 3개를 하나씩 (관점 이름, 검색어 틀)
            # ✅ TODO: query_template과 disease를 이용해 검색어(query) 생성
            query = query_template.format(disease=disease)     # "{disease} 증상과 진단 기준" → "고혈압 증상과 진단 기준"

            # ✅ TODO: tavily_tool로 검색 실행
            try:
                response = tavily_tool.invoke({"query": query})   # 검색 결과(딕셔너리)를 받음
            except Exception as e:                                # 일시적인 네트워크 오류로 1건이 실패해도
                print(f"[검색 실패] {query}: {e}")                  # 전체가 멈추지 않도록 알리고 건너뜀
                continue
            if not isinstance(response, dict) or "results" not in response:   # 결과가 0건이면 Tavily가 안내 문구(문자열)를 돌려줌
                print(f"[검색 결과 없음] {query}")                                # 알리고 다음 검색어로 넘어감
                continue

            # ✅ TODO: 검색 결과(response["results"])를 순회하며 raw_content(없으면 content)를 추출하여
            #          raw_items에 {"disease":.., "aspect":.., "raw_content":..} 형태로 추가
            for r in response["results"]:                                     # 검색 결과(최대 3개)를 하나씩
                content = r.get("raw_content") or r.get("content") or ""      # 원문이 없으면(None) 짧은 조각(content) 사용
                if content:                                                   # 내용이 있을 때만 저장
                    raw_items.append({"disease": disease,
                                      "aspect": aspect_name,
                                      "raw_content": content})

            print(f"[검색 완료] {disease} - {aspect_name}")

    return {"raw_items": raw_items}   # State의 raw_items 칸을 채워서 반환


# (확인용) 실제로 검색될 검색어를 API 호출 없이 문자열로만 만들어 봅니다
all_queries = [tpl.format(disease=d) for d in DISEASE_LIST for tpl in ASPECTS.values()]
print(f"총 {len(all_queries)}개 검색어 (질환 {len(DISEASE_LIST)}개 × 관점 {len(ASPECTS)}개)")
for q in all_queries[:6]:   # 앞의 6개만 미리 보기
    print(" -", q)
print(" ...")

총 18개 검색어 (질환 6개 × 관점 3개)
 - 고혈압 증상과 진단 기준
 - 고혈압 생활습관 관리 방법
 - 고혈압 복약 및 주의사항
 - 제2형 당뇨병 증상과 진단 기준
 - 제2형 당뇨병 생활습관 관리 방법
 - 제2형 당뇨병 복약 및 주의사항
 ...


## **(4) 정리 노드**

In [5]:
MAX_RAW_CHARS = 8000   # (추가) 웹 원문은 메뉴·광고까지 포함해 매우 길 수 있어서, 비용·시간을 줄이려고 앞 8,000자만 정제에 사용


def clean_node(state: PipelineState):
    clean_items = []   # 정제 결과를 모아 둘 리스트

    # ✅ TODO: 시스템 프롬프트(s_msg) 작성
    # - 웹 검색 원문에서 핵심 정보만 정제하는 역할 부여
    # - 질환·관점과 무관한 광고/메뉴/저작권 문구 등 노이즈 제거 지시
    # - 요약이 아닌, 관련 있는 내용은 그대로 유지하고 노이즈만 제거하도록 지시
    s_msg = SystemMessage("""당신은 의료 정보 데이터 정제 전문가입니다.
사용자가 웹 검색으로 수집한 원문과, 그 원문이 다루는 [질환]과 [관점]을 함께 제공합니다.

[해야 할 일]
- 해당 질환·관점과 관련된 의학 정보(증상, 진단 기준·수치, 생활습관 관리, 약 복용법, 부작용·주의사항 등)만 남기세요.
- 광고, 메뉴·내비게이션, 로그인·구독 안내, 저작권 문구, 댓글, 관련 기사 링크 등 질환·관점과 무관한 노이즈는 모두 제거하세요.

[하지 말아야 할 일]
- 요약하거나 문장을 줄이지 마세요. 관련 있는 내용은 원문 표현 그대로 유지하세요.
- 원문에 없는 내용을 추가하거나 추측해서 쓰지 마세요.

[출력 형식]
- 머리말이나 설명 없이 정제된 본문만 출력하세요.
- 관련 정보가 전혀 없으면 '관련 정보 없음'이라고만 출력하세요.
""")

    for item in state["raw_items"]:   # 검색 원문을 하나씩 정제
        # ✅ TODO: item의 disease, aspect, raw_content를 포함하여 HumanMessage(h_msg) 작성
        h_msg = HumanMessage(f"[질환] {item['disease']}\n[관점] {item['aspect']}\n\n"   # 어떤 질환·관점의 글인지 알려 주고
                             f"[원문]\n{item['raw_content'][:MAX_RAW_CHARS]}")           # 원문(앞 8,000자)을 붙임

        # ✅ TODO: llm.invoke([s_msg, h_msg])로 정제 결과 생성
        result = llm.invoke([s_msg, h_msg])                                   # GPT가 노이즈를 걷어낸 본문을 돌려줌
        clean_items.append({"disease": item["disease"], "aspect": item["aspect"],   # 질환·관점과 함께
                             "clean_content": result.content})                      # 정제 결과를 저장

    print(f"[정제 완료] 총 {len(clean_items)}개")
    return {"clean_items": clean_items}   # State의 clean_items 칸을 채워서 반환

## **(5) 그래프 구성 및 실행**

In [6]:
builder = StateGraph(PipelineState)            # PipelineState를 메모장으로 쓰는 그래프 설계도

# ✅ TODO: 노드 등록 (search_node, clean_node)
builder.add_node("search_node", search_node)   # 검색 담당 노드
builder.add_node("clean_node", clean_node)     # 정제 담당 노드

# ✅ TODO: 엣지 연결 (START → search_node → clean_node → END)
builder.add_edge(START, "search_node")         # 시작 → 검색
builder.add_edge("search_node", "clean_node")  # 검색 → 정제
builder.add_edge("clean_node", END)            # 정제 → 끝

# ✅ TODO: 그래프 컴파일
graph = builder.compile()                      # 설계도를 실행 가능한 그래프로 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → search_node 
search_node → clean_node 
clean_node → __end__ 


In [ ]:
# 🔑 [Colab에서 실행] 파이프라인 실행 — 검색 18회 + 정제 최대 54회라 몇 분 걸립니다 (Tavily·OpenAI 키 필요)
# ✅ TODO: PipelineState 초기값({"raw_items": [], "clean_items": []})을 전달하여 그래프 실행
result = graph.invoke({"raw_items": [], "clean_items": []})   # 빈 리스트로 시작 → 두 노드가 차례로 채움

print(f"수집된 원문: {len(result['raw_items'])}개")
print(f"정제된 항목: {len(result['clean_items'])}개")

# **3. 문서 구성**

* 정제된 항목(`clean_items`)을 질환별로 소제목과 함께 합쳐 하나의 텍스트 문서로 정리합니다.
* 이 단계는 그래프 밖에서 일반 코드로 처리합니다.

**💡 풀이 포인트** — 질환·관점마다 검색 결과가 최대 3개라서, 주석 그대로 `grouped[disease][aspect] = clean_content`처럼 대입만 하면 **마지막 결과만 남고 앞의 결과는 덮어써져 사라집니다.** 그래서 같은 칸에 이미 내용이 있으면 이어붙이도록 했고, 정제 결과가 '관련 정보 없음'인 항목은 문서에서 뺐습니다. `#`(질환명)·`##`(관점) 소제목을 붙여 두면 사람이 읽기도 좋고, 소제목 줄이 청크 안에 함께 들어가면 검색할 때 어떤 질환 이야기인지 단서가 됩니다.

In [ ]:
# 🔑 [Colab에서 실행] 위 파이프라인 실행 결과(result)가 있어야 동작합니다
aspect_order = ["증상과 진단 기준", "생활습관 관리 방법", "복약 및 주의사항"]   # 문서에 넣을 관점 순서

# ✅ TODO: clean_items를 질환별로 그룹화
#       grouped[disease][aspect] = clean_content
grouped = defaultdict(dict)   # grouped["고혈압"]처럼 처음 쓰는 질환 키도 자동으로 빈 딕셔너리가 만들어짐
for item in result["clean_items"]:
    content = item["clean_content"].strip()                        # 앞뒤 공백 제거
    if not content or content.startswith("관련 정보 없음"):          # 내용이 없거나 관련 정보가 없으면
        continue                                                   # 문서에 넣지 않음
    disease, aspect = item["disease"], item["aspect"]              # 질환명, 관점 꺼내기
    if aspect in grouped[disease]:                                 # 같은 질환·관점에 이미 내용이 있으면
        grouped[disease][aspect] += "\n\n" + content               # 덮어쓰지 않고 뒤에 이어붙이기
    else:
        grouped[disease][aspect] = content                         # 처음이면 그대로 저장

# ✅ TODO: 질환별 소제목 구조로 결합 (doc_parts 리스트에 "# 질환명", "## 관점\n내용" 형태로 추가)
doc_parts = []                                                     # 문서 조각(제목·내용)을 순서대로 담을 리스트
for disease in DISEASE_LIST:                                       # 정해 둔 질환 순서대로
    if disease not in grouped:                                     # 정보가 하나도 없는 질환은 건너뜀
        continue
    doc_parts.append(f"# {disease}")                               # 대제목: 질환명
    for aspect in aspect_order:                                    # 정해 둔 관점 순서대로
        if aspect in grouped[disease]:
            doc_parts.append(f"## {aspect}\n{grouped[disease][aspect]}")   # 소제목 + 내용

full_document = "\n".join(doc_parts)   # 조각들을 줄바꿈으로 이어 하나의 문서로

# ✅ TODO: full_document를 "만성질환_정보.txt" 파일로 저장 (encoding="utf-8")
with open("만성질환_정보.txt", "w", encoding="utf-8") as f:   # 한글이 깨지지 않게 utf-8로 저장
    f.write(full_document)                                    # 파일에 쓰기

print("문서 저장 완료, 총 길이:", len(full_document))

# **4. 벡터DB 구축**

**💡 풀이 포인트** — **청크**(chunk)는 긴 문서를 검색하기 좋은 크기로 자른 조각이고, **임베딩**(embedding)은 문장의 의미를 숫자 목록(벡터)으로 바꾸는 것입니다. 의미가 비슷한 문장은 벡터도 가까워서, 질문과 '가까운' 청크를 찾는 방식으로 검색합니다(**벡터DB**는 이 벡터들을 저장해 두고 빨리 찾아주는 DB이고, 여기서는 Chroma를 씁니다). `chunk_size`가 너무 작으면 문맥이 끊기고 너무 크면 검색이 덜 정밀해져서 1,000자로 잡고, 경계에서 내용이 끊기지 않게 앞뒤 청크가 최대 100자 정도 겹치도록(줄 단위로 겹침) 했습니다. ⚠️ 5·6일차 노트북이 `text-embedding-3-small` 모델로 이 DB를 불러오기 때문에, 만들 때도 **반드시 같은 임베딩 모델**을 써야 합니다(모델이 다르면 같은 문장도 전혀 다른 숫자가 되어 검색이 엉뚱해집니다).

## **(1) 분할 (Split)**

In [ ]:
# 🔑 [Colab에서 실행] 위에서 저장한 "만성질환_정보.txt"가 있어야 동작합니다
# ✅ TODO: TextLoader로 "만성질환_정보.txt" 로딩
txt_loader = TextLoader("만성질환_정보.txt", encoding="utf-8")   # 한글 파일이라 encoding을 지정
txt_docs = txt_loader.load()                                      # 파일 1개 → Document 1개가 든 리스트

# ✅ TODO: CharacterTextSplitter로 청크 분할 (chunk_size, chunk_overlap, separator 설정)
text_splitter = CharacterTextSplitter(
    separator="\n",       # 줄바꿈 위치에서 자르기 (문장·항목 중간이 잘리지 않게)
    chunk_size=1000,      # 청크 하나의 최대 글자 수 (한 줄이 이보다 길면 그 줄은 통째로 한 청크가 됨)
    chunk_overlap=100,    # 앞뒤 청크가 최대 100자 정도(줄 단위로) 겹치게 → 경계에서 문맥이 끊기는 것 방지
)

split_docs = text_splitter.split_documents(txt_docs)   # Document를 청크 여러 개로 자르기
print("청크 개수:", len(split_docs))

## **(2) 임베딩 & 벡터DB 저장**

In [ ]:
# 🔑 [Colab에서 실행] 임베딩 API를 호출합니다 (OpenAI 키 필요)
# ✅ TODO: 임베딩 모델 생성 (OpenAIEmbeddings)
#   ⚠️ 5·6일차가 text-embedding-3-small로 DB를 불러오므로 같은 모델을 지정해야 합니다
embedding = OpenAIEmbeddings(model="text-embedding-3-small")   # 문장 → 벡터 변환 모델

# (추가) 셀을 다시 실행하면 같은 청크가 DB에 중복 저장되므로, 기존 컬렉션이 있으면 먼저 비웁니다
Chroma(persist_directory="./chronic_db", embedding_function=embedding).delete_collection()   # 처음 실행이면 빈 컬렉션을 만들었다가 지우는 것뿐이라 안전

# ✅ TODO: Chroma.from_documents로 벡터DB 생성 및 저장 (persist_directory="./chronic_db")
vectorstore = Chroma.from_documents(   # 청크를 임베딩해서 DB에 저장
    documents=split_docs,               # 저장할 청크들
    embedding=embedding,                # 청크 → 벡터로 바꿔 줄 임베딩 모델
    persist_directory="./chronic_db",   # 이 폴더에 DB 파일이 저장됨 (5·6일차에서 재사용)
)
print("저장된 청크 수:", vectorstore._collection.count())   # 위의 청크 개수와 같아야 정상

# **5. 검증**

In [ ]:
# 🔑 [Colab에서 실행] 질문도 임베딩해서 검색하므로 OpenAI 키 필요
# ✅ TODO: 질환 관련 질문(query) 작성
query = "고혈압 환자가 피해야 할 음식은 무엇인가요?"   # 검증용 질문

# ✅ TODO: vectorstore.similarity_search로 유사도 검색 실행 (k=3)
search_results = vectorstore.similarity_search(query, k=3)   # 질문과 의미가 가장 가까운 청크 3개

for i, doc in enumerate(search_results):
    print(f"--- 검색결과 {i+1} ---")
    print(doc.page_content[:300])
    print()

# **6. (추가) 5·6일차를 위한 저장**

* 5일차 노트북은 구글 드라이브 `ai_agent` 폴더의 `chronic_db`를 불러옵니다. Colab의 `/content` 폴더는 런타임이 끝나면 지워지므로, 만든 DB와 원문 문서를 드라이브에 복사해 둡니다.
* 6일차 노트북은 `chronic_db`가 노트북과 같은 위치(`/content`)에 있어야 하는데, 6일차 노트북에 드라이브에서 복사해 오는 셀을 넣어 두었습니다.

In [ ]:
# 🔑 [Colab에서 실행] 구글 드라이브에 결과물 복사 (Colab 전용)
import shutil                                   # 폴더·파일 복사 도구
from google.colab import drive                  # 구글 드라이브 연결 도구
drive.mount('/content/drive')                   # 드라이브 연결 (처음엔 권한 허용 창이 뜸)

save_dir = '/content/drive/MyDrive/ai_agent/'   # 5일차 노트북이 사용하는 폴더
os.makedirs(save_dir, exist_ok=True)            # 폴더가 없으면 새로 만듦
shutil.copytree('./chronic_db', save_dir + 'chronic_db', dirs_exist_ok=True)   # 벡터DB 폴더 통째로 복사
shutil.copy('만성질환_정보.txt', save_dir)                                       # 원문 문서도 함께 보관
print("복사 완료:", os.listdir(save_dir))